In [ ]:
# ============================================================
# BRANCH A — DIRECT DOCUMENT REPRESENTATION
# D6 — Microsoft FY24 Q1 Press Release
# ============================================================
#
# Methodology stages covered:
# Stage 2 — Branch A: Direct Ingestion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================

from google.colab import files
from pathlib import Path
from collections import Counter

import hashlib
import json
import platform
import re
import sys

import pandas as pd


In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D6"

DOCUMENT_NAME = (
    "Microsoft FY24 Q1 Press Release"
)

BRANCH = "A"

BRANCH_NAME = "Direct Ingestion"

SOURCE_FORMAT = ".pdf"

INPUT_REPRESENTATION = "Original PDF document"

DIRECT_DOCUMENT_INGESTION = True

EXPECTED_PAGE_COUNT = 10

EXPECTED_RECORD_COUNT = 147

EXPECTED_CATEGORY_COUNTS = {
    "Narrative performance highlight": 24,
    "Financial performance reconciliation": 4,
    "Segment revenue reconciliation": 3,
    "Selected product and service reconciliation": 15,
    "Income statement": 19,
    "Comprehensive income statement": 6,
    "Balance sheet": 34,
    "Cash flow statement": 34,
    "Segment revenue and operating income": 8
}

EXPECTED_PART_COUNTS = {
    1: 46,
    2: 25,
    3: 34,
    4: 34,
    5: 8
}


EXPECTED_FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Business Area",
    "Value 2023",
    "Value 2022",
    "GAAP YoY Change",
    "Constant Currency Impact",
    "Constant Currency YoY Change",
    "Unit",
    "Reporting Period",
    "Source Location"
]


STRING_OR_NULL_FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Business Area",
    "Unit",
    "Reporting Period",
    "Source Location"
]


NUMERIC_OR_NULL_FIELDS = [
    "Value 2023",
    "Value 2022",
    "GAAP YoY Change",
    "Constant Currency Impact",
    "Constant Currency YoY Change"
]


MANDATORY_CONTENT_FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Business Area",
    "Unit",
    "Reporting Period",
    "Source Location"
]


ALLOWED_CATEGORIES = set(
    EXPECTED_CATEGORY_COUNTS
)


OUTPUT_DIR = Path(
    "outputs_D6_branch_A"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


INPUT_INTEGRITY_PATH = (
    OUTPUT_DIR
    / "D6_branch_A_input_integrity.json"
)

REPRESENTATION_PATH = (
    OUTPUT_DIR
    / "D6_branch_A_representation.json"
)

PART_EXECUTION_SUMMARY_PATH = (
    OUTPUT_DIR
    / "D6_branch_A_part_execution_summary.json"
)

COMBINED_EXTRACTION_PATH = (
    OUTPUT_DIR
    / "D6_branch_A_combined_parsed_extraction.json"
)

TECHNICAL_DIAGNOSTICS_PATH = (
    OUTPUT_DIR
    / "D6_branch_A_technical_diagnostics.json"
)

EXPERIMENT_METADATA_PATH = (
    OUTPUT_DIR
    / "D6_branch_A_experiment_metadata.json"
)

EXPERIMENT_SUMMARY_PATH = (
    OUTPUT_DIR
    / "D6_branch_A_experiment_summary.json"
)


print(
    "Document:",
    DOCUMENT_ID
)

print(
    "Branch:",
    BRANCH
)

print(
    "Input representation:",
    INPUT_REPRESENTATION
)

print(
    "Expected records:",
    EXPECTED_RECORD_COUNT
)

print(
    "Output directory:",
    OUTPUT_DIR
)

In [ ]:
# ============================================================
# 2. Source document upload
# ============================================================

print("Upload the original D6 Microsoft FY24 Q1 Press Release PDF.")

uploaded = files.upload()

pdf_files = [
    Path(filename)
    for filename in uploaded.keys()
    if filename.lower().endswith(".pdf")
]

if len(pdf_files) != 1:
    raise ValueError("Upload exactly one PDF file.")

SOURCE_PATH = pdf_files[0]

print("Source file:", SOURCE_PATH.name)

In [ ]:
# ============================================================
# 3. Source SHA-256
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


SOURCE_SHA256 = sha256_file(SOURCE_PATH)

print("Source SHA-256:", SOURCE_SHA256)


In [ ]:
# ============================================================
# 4. Source PDF diagnostics
# ============================================================

try:
    import fitz
except ImportError:
    !pip -q install pymupdf
    import fitz

pdf_document = fitz.open(SOURCE_PATH)

PAGE_COUNT = len(pdf_document)
PAGE_COUNT_VALID = PAGE_COUNT == EXPECTED_PAGE_COUNT

page_characterisation_rows = []

for page_number, page in enumerate(pdf_document, start=1):
    page_text = page.get_text("text") or ""

    page_characterisation_rows.append(
        {
            "Page Number": page_number,
            "Character Count": len(page_text),
            "Word Count": len(page_text.split()),
            "Text Extractable": bool(page_text.strip())
        }
    )

page_characterisation_df = pd.DataFrame(page_characterisation_rows)

TOTAL_TEXT_CHARACTERS = int(
    page_characterisation_df["Character Count"].sum()
)

TEXT_EXTRACTABLE = bool(
    page_characterisation_df["Text Extractable"].all()
)

OCR_REQUIRED = not TEXT_EXTRACTABLE

print("Page count:", PAGE_COUNT)
print("Page count valid:", PAGE_COUNT_VALID)
print("Text extractable:", TEXT_EXTRACTABLE)
print("OCR required:", OCR_REQUIRED)

display(page_characterisation_df)


In [ ]:
# ============================================================
# 5. Source grounding verification
# ============================================================

full_text = "\n".join(
    page.get_text("text") or ""
    for page in pdf_document
)

GROUNDING_MARKERS = {
    "quarterly_results": "Revenue was $56.5 billion",
    "business_highlights": "Business Highlights",
    "constant_currency_reconciliation":
        "Financial Performance Constant Currency Reconciliation",
    "income_statements": "INCOME STATEMENTS",
    "comprehensive_income_statements":
        "COMPREHENSIVE INCOME STATEMENTS",
    "balance_sheets": "BALANCE SHEETS",
    "cash_flows_statements": "CASH FLOWS STATEMENTS",
    "segment_revenue_and_operating_income":
        "SEGMENT REVENUE AND OPERATING INCOME"
}

GROUNDING_MARKER_STATUS = {
    marker_name: marker_text.casefold() in full_text.casefold()
    for marker_name, marker_text in GROUNDING_MARKERS.items()
}

DOCUMENT_GROUNDING_VALID = all(GROUNDING_MARKER_STATUS.values())

print(json.dumps(GROUNDING_MARKER_STATUS, indent=2))
print("Document grounding valid:", DOCUMENT_GROUNDING_VALID)

if not PAGE_COUNT_VALID:
    raise AssertionError(
        f"Expected {EXPECTED_PAGE_COUNT} pages, found {PAGE_COUNT}."
    )

if not TEXT_EXTRACTABLE:
    raise AssertionError(
        "The uploaded PDF does not contain extractable text."
    )

if not DOCUMENT_GROUNDING_VALID:
    raise AssertionError(
        "One or more expected D6 source sections were not detected."
    )


In [ ]:
# ============================================================
# 6. Source integrity report
# ============================================================

FILE_SIZE_BYTES = SOURCE_PATH.stat().st_size

FILE_NON_EMPTY = (
    FILE_SIZE_BYTES > 0
)

INPUT_INTEGRITY_PASSED = all([
    FILE_NON_EMPTY,
    PAGE_COUNT_VALID,
    TEXT_EXTRACTABLE,
    DOCUMENT_GROUNDING_VALID
])


INPUT_INTEGRITY = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_file":
        SOURCE_PATH.name,

    "input_file_sha256":
        SOURCE_SHA256,

    "input_representation":
        INPUT_REPRESENTATION,

    "file_size_bytes":
        FILE_SIZE_BYTES,

    "file_non_empty":
        FILE_NON_EMPTY,

    "page_count":
        PAGE_COUNT,

    "expected_page_count":
        EXPECTED_PAGE_COUNT,

    "page_count_valid":
        PAGE_COUNT_VALID,

    "text_layer_present":
        TEXT_EXTRACTABLE,

    "ocr_required":
        OCR_REQUIRED,

    "grounding_marker_checks":
        GROUNDING_MARKER_STATUS,

    "all_expected_components_present":
        DOCUMENT_GROUNDING_VALID,

    "direct_pdf_ingestion_usable":
        bool(
            PAGE_COUNT_VALID
            and TEXT_EXTRACTABLE
            and DOCUMENT_GROUNDING_VALID
        ),

    "input_integrity_passed":
        bool(
            INPUT_INTEGRITY_PASSED
        )
}


INPUT_INTEGRITY_PATH.write_text(
    json.dumps(
        INPUT_INTEGRITY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        INPUT_INTEGRITY,
        ensure_ascii=False,
        indent=2
    )
)

In [ ]:
# ============================================================
# 7. Branch A representation
# ============================================================

REPRESENTATION = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "representation_type":
        "Original source document",

    "input_file":
        SOURCE_PATH.name,

    "input_format":
        SOURCE_FORMAT,

    "diagnostic_pdf_text_inspection_applied":
        True,

    "pdf_to_text_conversion_applied":
        False,

    "derived_representation_used_as_model_input":
        False,

    "ocr_applied":
        False,

    "page_cropping_applied":
        False,

    "page_extraction_applied":
        False,

    "layout_reconstruction_applied":
        False,

    "table_conversion_applied":
        False,

    "structural_conversion_applied":
        False,

    "normalisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_conversion_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "source_content_modification_applied":
        False,

    "split_extraction_applied":
        True,

    "split_part_count":
        5,

    "model_input_description": (
        "Each extraction part receives the complete original "
        "ten-page PDF. PyMuPDF text extraction is used only "
        "for source-integrity diagnostics and is not supplied "
        "to the model as an alternative representation."
    )
}


REPRESENTATION_PATH.write_text(
    json.dumps(
        REPRESENTATION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        REPRESENTATION,
        ensure_ascii=False,
        indent=2
    )
)

In [ ]:
# ============================================================
# 8. Split extraction prompts
# ============================================================

COMMON_INSTRUCTIONS = r'''
You are an information extraction assistant.

Extract the requested financial and quantitative records represented
within the defined source scope of the attached original Microsoft FY24
Q1 Press Release PDF.

Treat the attached original PDF as the only source of information.

For every included record return exactly these twelve fields:

- Category
- Statement or Section
- Metric
- Business Area
- Value 2023
- Value 2022
- GAAP YoY Change
- Constant Currency Impact
- Constant Currency YoY Change
- Unit
- Reporting Period
- Source Location

General extraction rules:

- Use JSON numbers for explicitly represented numeric values.
- Use JSON null when a field is not explicitly represented.
- Preserve negative values.
- Preserve explicit zero values.
- Preserve the source measurement scale.
- Preserve repeated observations when they occur in distinct source
  sections.
- Do not calculate or infer values.
- Do not derive prior-period values from growth rates.
- Do not convert units.
- Do not normalise measurement scales.
- Do not silently correct source values.
- Do not use external knowledge.
- Do not include publication metadata, webcast details, contact details,
  URLs, qualitative outlook text or forward-looking risk narrative.
- Do not include values that appear only inside descriptive accounting
  line-item labels when they are not primary observations.

For Balance Sheets only:

- place the September 30, 2023 amount in "Value 2023";
- place the June 30, 2023 amount in "Value 2022".

Return only valid JSON.

Do not include Markdown fences, explanations or commentary.

Return the result using exactly this top-level structure:

{
  "document_id": "D6",
  "branch": "A",
  "part": PART_NUMBER,
  "records": [
    {
      "Category": null,
      "Statement or Section": null,
      "Metric": null,
      "Business Area": null,
      "Value 2023": null,
      "Value 2022": null,
      "GAAP YoY Change": null,
      "Constant Currency Impact": null,
      "Constant Currency YoY Change": null,
      "Unit": null,
      "Reporting Period": null,
      "Source Location": null
    }
  ]
}

Return only the JSON object.
'''.strip()


PART_SCOPES = {

    1: r'''
PART 1 SOURCE SCOPE

Include only:

1. The quantitative company-performance observations represented in the
   quarterly-results and business-highlights sections on page 1 and the
   shareholder-return observation on page 2.

2. Every represented metric in the following three constant-currency
   reconciliation tables on page 3:

   - Financial Performance Constant Currency Reconciliation
   - Segment Revenue Constant Currency Reconciliation
   - Selected Product and Service Revenue Constant Currency Reconciliation

Valid Category values for this part:

- Narrative performance highlight
- Financial performance reconciliation
- Segment revenue reconciliation
- Selected product and service reconciliation

Valid Source Location values:

- Page 1 — Quarterly results
- Page 1 — Business Highlights
- Page 2 — Shareholder returns
- Page 3 — Financial Performance Constant Currency Reconciliation
- Page 3 — Segment Revenue Constant Currency Reconciliation
- Page 3 — Selected Product and Service Revenue Constant Currency Reconciliation

Extract every record represented within this defined source scope.
''',

    2: r'''
PART 2 SOURCE SCOPE

Include every primary financial line item represented in:

- Income Statements on page 6;
- Comprehensive Income Statements on page 7.

Exclude section headings and accounting-label values that are not
primary financial line-item observations.

Valid Category values:

- Income statement
- Comprehensive income statement

Valid Source Location values:

- Page 6 — Income Statements
- Page 7 — Comprehensive Income Statements

Use:

- "Income Statements" or "Comprehensive Income Statements" for
  Statement or Section;
- "Corporate" for Business Area;
- "Three months ended September 30" for Reporting Period.

Extract every record represented within this defined source scope.
''',

    3: r'''
PART 3 SOURCE SCOPE

Include every primary Balance Sheets line item represented on page 8.

Exclude:

- headings without amounts;
- Commitments and contingencies;
- allowance amounts embedded only in the accounts-receivable label;
- accumulated depreciation amounts embedded only in the property and
  equipment label;
- authorised and outstanding share quantities embedded only in the
  common-stock label.

Use:

- Category: "Balance sheet"
- Statement or Section: "Balance Sheets"
- Business Area: "Corporate"
- Unit: "USD millions"
- Reporting Period: "September 30, 2023 and June 30, 2023"
- Source Location: "Page 8 — Balance Sheets"

Place September 30, 2023 values in "Value 2023".

Place June 30, 2023 values in "Value 2022".

Extract every record represented within this defined source scope.
''',

    4: r'''
PART 4 SOURCE SCOPE

Include every primary Cash Flows Statements line item represented on
page 9.

Preserve parentheses as negative values and preserve explicit zeros.

Distinguish the two separate source observations:

- Other, net — financing
- Other, net — investing

Use:

- Category: "Cash flow statement"
- Statement or Section: "Cash Flows Statements"
- Business Area: "Corporate"
- Unit: "USD millions"
- Reporting Period: "Three months ended September 30"
- Source Location: "Page 9 — Cash Flows Statements"

Extract every record represented within this defined source scope.
''',

    5: r'''
PART 5 SOURCE SCOPE

Include every represented revenue and operating-income row from the
Segment Revenue and Operating Income table on page 10.

The table contains revenue observations and operating-income
observations for the represented business segments and total rows.

Use:

- Category: "Segment revenue and operating income"
- Statement or Section: "Segment Revenue and Operating Income"
- Unit: "USD millions"
- Reporting Period: "Three months ended September 30"
- Source Location: "Page 10 — Segment Revenue and Operating Income"

Extract every record represented within this defined source scope.
'''
}


PART_PROMPT_PATHS = {}


for part_number, part_scope in PART_SCOPES.items():

    common_for_part = COMMON_INSTRUCTIONS.replace(
        "PART_NUMBER",
        str(
            part_number
        )
    )

    part_prompt = (
        common_for_part
        + "\n\n"
        + part_scope.strip()
    )


    part_path = (
        OUTPUT_DIR
        / f"D6_branch_A_prompt_part_{part_number}.txt"
    )


    part_path.write_text(
        part_prompt,
        encoding="utf-8"
    )


    PART_PROMPT_PATHS[
        part_number
    ] = part_path


    print(
        f"Part {part_number}:",
        part_path.name,
        "| SHA-256:",
        sha256_file(
            part_path
        )
    )

## Independent Branch A extraction

Five independent model executions.

For each part:

1. Open a new independent conversation.
2. Upload complete original D6 PDF.
3. Upload corresponding prompt:
   - `D6_branch_A_prompt_part_1.txt`
   - `D6_branch_A_prompt_part_2.txt`
   - `D6_branch_A_prompt_part_3.txt`
   - `D6_branch_A_prompt_part_4.txt`
   - `D6_branch_A_prompt_part_5.txt`
4. Submit the corresponding prompt once.
5. Save the complete unmodified model response as:
   - `D6_branch_A_raw_response_part_1.txt`
   - `D6_branch_A_raw_response_part_2.txt`
   - `D6_branch_A_raw_response_part_3.txt`
   - `D6_branch_A_raw_response_part_4.txt`
   - `D6_branch_A_raw_response_part_5.txt`

Upload the five untouched TXT responses in the next cell.

In [ ]:
# ============================================================
# 9. Raw-response upload and preservation
# ============================================================

print(
    "Upload exactly five TXT files:\n"
    "- D6_branch_A_raw_response_part_1.txt\n"
    "- D6_branch_A_raw_response_part_2.txt\n"
    "- D6_branch_A_raw_response_part_3.txt\n"
    "- D6_branch_A_raw_response_part_4.txt\n"
    "- D6_branch_A_raw_response_part_5.txt"
)


uploaded = files.upload()


uploaded_txt_paths = [
    Path(
        filename
    )

    for filename in uploaded.keys()

    if filename.lower().endswith(
        ".txt"
    )
]


if len(
    uploaded_txt_paths
) != 5:

    raise ValueError(
        "Upload exactly five TXT raw-response files."
    )


def detect_part_number(
    path
):

    match = re.search(
        r"part[_\- ]?([1-5])",
        path.stem,
        flags=re.IGNORECASE
    )

    if match is None:

        raise ValueError(
            "Could not detect a part number from "
            f"{path.name}"
        )

    return int(
        match.group(
            1
        )
    )


PART_RAW_RESPONSE_PATHS = {}


for uploaded_path in uploaded_txt_paths:

    part_number = detect_part_number(
        uploaded_path
    )

    if (
        part_number
        in PART_RAW_RESPONSE_PATHS
    ):

        raise ValueError(
            f"More than one response was detected "
            f"for part {part_number}."
        )


    raw_text = uploaded_path.read_text(
        encoding="utf-8"
    )


    if not raw_text.strip():

        raise ValueError(
            f"Raw response for part {part_number} is empty."
        )


    preserved_path = (
        OUTPUT_DIR
        / (
            "D6_branch_A_raw_response_"
            f"part_{part_number}.txt"
        )
    )


    preserved_path.write_text(
        raw_text,
        encoding="utf-8"
    )


    PART_RAW_RESPONSE_PATHS[
        part_number
    ] = preserved_path


if (
    set(
        PART_RAW_RESPONSE_PATHS
    )
    != set(
        EXPECTED_PART_COUNTS
    )
):

    raise ValueError(
        "The uploaded files must represent parts "
        "1, 2, 3, 4 and 5."
    )


PART_RAW_RESPONSE_METADATA = {
    str(
        part_number
    ): {
        "file":
            path.name,

        "sha256":
            sha256_file(
                path
            ),

        "character_count":
            len(
                path.read_text(
                    encoding="utf-8"
                )
            )
    }

    for part_number, path
    in sorted(
        PART_RAW_RESPONSE_PATHS.items()
    )
}


print(
    json.dumps(
        PART_RAW_RESPONSE_METADATA,
        ensure_ascii=False,
        indent=2
    )
)

In [ ]:
# ============================================================
# 10. Raw-response parsing
# ============================================================

part_parsing_results = {}

part_records = {}

combined_records = []

all_parts_json_valid = True

all_parts_records_evaluable = True


for part_number in sorted(
    PART_RAW_RESPONSE_PATHS
):

    raw_path = (
        PART_RAW_RESPONSE_PATHS[
            part_number
        ]
    )

    raw_text = raw_path.read_text(
        encoding="utf-8"
    )


    valid_json = False

    parsing_error = None

    parsed_part = None


    try:

        parsed_part = json.loads(
            raw_text
        )

        valid_json = True


    except json.JSONDecodeError as error:

        parsing_error = str(
            error
        )


    top_level_object_valid = (
        valid_json
        and isinstance(
            parsed_part,
            dict
        )
    )


    document_id_present = (
        top_level_object_valid
        and "document_id"
        in parsed_part
    )

    document_id_correct = (
        top_level_object_valid
        and parsed_part.get(
            "document_id"
        )
        == DOCUMENT_ID
    )


    branch_present = (
        top_level_object_valid
        and "branch"
        in parsed_part
    )

    branch_correct = (
        top_level_object_valid
        and parsed_part.get(
            "branch"
        )
        == BRANCH
    )


    part_present = (
        top_level_object_valid
        and "part"
        in parsed_part
    )

    part_correct = (
        top_level_object_valid
        and parsed_part.get(
            "part"
        )
        == part_number
    )


    records_present = (
        top_level_object_valid
        and "records"
        in parsed_part
    )

    records_is_list = (
        top_level_object_valid
        and isinstance(
            parsed_part.get(
                "records"
            ),
            list
        )
    )


    records_evaluable = all([
        valid_json,
        top_level_object_valid,
        document_id_present,
        document_id_correct,
        branch_present,
        branch_correct,
        part_present,
        part_correct,
        records_present,
        records_is_list
    ])


    if records_evaluable:

        records = parsed_part[
            "records"
        ]

    else:

        records = []


    part_records[
        part_number
    ] = records


    observed_part_count = (
        len(
            records
        )
        if records_evaluable
        else None
    )


    expected_part_count = (
        EXPECTED_PART_COUNTS[
            part_number
        ]
    )


    part_record_count_matches = (
        observed_part_count
        == expected_part_count

        if records_evaluable

        else None
    )


    part_parsing_results[
        part_number
    ] = {
        "raw_response_file":
            raw_path.name,

        "raw_response_sha256":
            sha256_file(
                raw_path
            ),

        "json_valid":
            valid_json,

        "json_parsing_error":
            parsing_error,

        "top_level_object_valid":
            top_level_object_valid,

        "document_id_present":
            document_id_present,

        "document_id_correct":
            document_id_correct,

        "branch_present":
            branch_present,

        "branch_correct":
            branch_correct,

        "part_present":
            part_present,

        "part_correct":
            part_correct,

        "records_present":
            records_present,

        "records_is_list":
            records_is_list,

        "records_evaluable":
            records_evaluable,

        "expected_record_count":
            expected_part_count,

        "observed_record_count":
            observed_part_count,

        "record_count_matches":
            part_record_count_matches
    }


    if records_evaluable:

        combined_records.extend(
            records
        )


    if not valid_json:

        all_parts_json_valid = False


    if not records_evaluable:

        all_parts_records_evaluable = False


extracted_records = (
    combined_records
)


observed_record_count = (
    len(
        extracted_records
    )
)


PART_EXECUTION_SUMMARY = {
    str(
        part_number
    ):
        result

    for part_number, result
    in part_parsing_results.items()
}


PART_EXECUTION_SUMMARY_PATH.write_text(
    json.dumps(
        PART_EXECUTION_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

combined_extraction_created = False

combined_extraction_sha256 = None


if all_parts_records_evaluable:

    COMBINED_EXTRACTION = {
        "document_id":
            DOCUMENT_ID,

        "branch":
            BRANCH,

        "records":
            extracted_records
    }


    COMBINED_EXTRACTION_PATH.write_text(
        json.dumps(
            COMBINED_EXTRACTION,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )


    combined_extraction_created = True

    combined_extraction_sha256 = (
        sha256_file(
            COMBINED_EXTRACTION_PATH
        )
    )


print(
    "Part parsing results:"
)

print(
    json.dumps(
        PART_EXECUTION_SUMMARY,
        ensure_ascii=False,
        indent=2
    )
)

print(
    "\nAll parts valid JSON:",
    all_parts_json_valid
)

print(
    "All parts records evaluable:",
    all_parts_records_evaluable
)

print(
    "Combined observed records:",
    (
        observed_record_count
        if all_parts_records_evaluable
        else "Not fully evaluable"
    )
)

In [ ]:
# ============================================================
# 11. Record-structure diagnostics
# ============================================================

record_structure_issues = []


if all_parts_records_evaluable:

    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(
            record,
            dict
        ):

            record_structure_issues.append(
                {
                    "record_index":
                        record_index,

                    "issue":
                        "Record is not a JSON object"
                }
            )

            continue


        observed_fields = list(
            record.keys()
        )


        if (
            observed_fields
            != EXPECTED_FIELDS
        ):

            record_structure_issues.append(
                {
                    "record_index":
                        record_index,

                    "issue":
                        "Field names or field order differ",

                    "expected_fields":
                        EXPECTED_FIELDS,

                    "observed_fields":
                        observed_fields
                }
            )


    records_with_structure_issues = len({
        issue[
            "record_index"
        ]

        for issue
        in record_structure_issues
    })


    record_schema_valid = (
        records_with_structure_issues
        == 0
    )


else:

    records_with_structure_issues = None

    record_schema_valid = None


print(
    "Record schema valid:",
    record_schema_valid
)

print(
    "Records with structure issues:",
    records_with_structure_issues
)

In [ ]:
# ============================================================
# 12. Field-type and mandatory-content diagnostics
# ============================================================

field_type_issues = []

missing_mandatory_values = []


if all_parts_records_evaluable:

    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(
            record,
            dict
        ):

            continue

        for field in STRING_OR_NULL_FIELDS:

            value = record.get(
                field
            )

            if (
                value is not None
                and not isinstance(
                    value,
                    str
                )
            ):

                field_type_issues.append(
                    {
                        "record_index":
                            record_index,

                        "field":
                            field,

                        "observed_type":
                            type(
                                value
                            ).__name__,

                        "expected_type":
                            "string or null"
                    }
                )

        for field in NUMERIC_OR_NULL_FIELDS:

            value = record.get(
                field
            )

            if (
                isinstance(
                    value,
                    bool
                )
                or (
                    value is not None
                    and not isinstance(
                        value,
                        (
                            int,
                            float
                        )
                    )
                )
            ):

                field_type_issues.append(
                    {
                        "record_index":
                            record_index,

                        "field":
                            field,

                        "observed_type":
                            type(
                                value
                            ).__name__,

                        "expected_type":
                            "number or null"
                    }
                )

        for field in MANDATORY_CONTENT_FIELDS:

            value = record.get(
                field
            )

            if (
                value is None
                or value == ""
            ):

                missing_mandatory_values.append(
                    {
                        "record_index":
                            record_index,

                        "field":
                            field
                    }
                )


    records_with_type_issues = len({
        issue[
            "record_index"
        ]

        for issue
        in field_type_issues
    })


    field_types_valid = (
        records_with_type_issues
        == 0
    )


    missing_mandatory_value_count = len(
        missing_mandatory_values
    )


    mandatory_fields_complete = (
        missing_mandatory_value_count
        == 0
    )


else:

    records_with_type_issues = None

    field_types_valid = None

    missing_mandatory_value_count = None

    mandatory_fields_complete = None


print(
    "Records with type issues:",
    records_with_type_issues
)

print(
    "Field types valid:",
    field_types_valid
)

print(
    "Missing mandatory value count:",
    missing_mandatory_value_count
)

print(
    "Mandatory fields complete:",
    mandatory_fields_complete
)

In [ ]:
# ============================================================
# 13. Category and count diagnostics
# ============================================================

if all_parts_records_evaluable:

    observed_record_count = len(
        extracted_records
    )


    record_count_valid = (
        observed_record_count
        == EXPECTED_RECORD_COUNT
    )


    observed_category_counts = dict(
        Counter(
            record.get(
                "Category"
            )

            for record
            in extracted_records

            if isinstance(
                record,
                dict
            )
        )
    )


    categories_valid = set(
        observed_category_counts
    ).issubset(
        ALLOWED_CATEGORIES
    )


    category_counts_valid = (
        observed_category_counts
        == EXPECTED_CATEGORY_COUNTS
    )


else:

    observed_record_count = None

    record_count_valid = None

    observed_category_counts = None

    categories_valid = None

    category_counts_valid = None


print(
    "Expected records:",
    EXPECTED_RECORD_COUNT
)

print(
    "Observed records:",
    observed_record_count
)

print(
    "Record count matches:",
    record_count_valid
)

print(
    "Categories valid:",
    categories_valid
)

print(
    "Category counts match:",
    category_counts_valid
)

In [ ]:
# ============================================================
# 14. Duplicate-record diagnostics
# ============================================================

DUPLICATE_KEY_FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Business Area",
    "Reporting Period",
    "Source Location"
]


if all_parts_records_evaluable:

    duplicate_counter = Counter()


    for record in extracted_records:

        if not isinstance(
            record,
            dict
        ):

            continue


        duplicate_key = tuple(
            record.get(
                field
            )

            for field
            in DUPLICATE_KEY_FIELDS
        )


        duplicate_counter[
            duplicate_key
        ] += 1


    duplicate_record_keys = [
        list(
            key
        )

        for key, count
        in duplicate_counter.items()

        if count > 1
    ]


    duplicate_record_key_count = len(
        duplicate_record_keys
    )


    duplicate_record_keys_absent = (
        duplicate_record_key_count
        == 0
    )


else:

    duplicate_record_keys = None

    duplicate_record_key_count = None

    duplicate_record_keys_absent = None


print(
    "Duplicate record key count:",
    duplicate_record_key_count
)

In [ ]:
# ============================================================
# 15. Content diagnostics
# ============================================================

if all_parts_records_evaluable:

    numeric_values = []


    for record in extracted_records:

        if not isinstance(
            record,
            dict
        ):

            continue


        for field in NUMERIC_OR_NULL_FIELDS:

            value = record.get(
                field
            )

            if (
                isinstance(
                    value,
                    (
                        int,
                        float
                    )
                )
                and not isinstance(
                    value,
                    bool
                )
            ):

                numeric_values.append(
                    value
                )


    negative_value_count = sum(
        value < 0

        for value in numeric_values
    )


    zero_value_count = sum(
        value == 0

        for value in numeric_values
    )


    negative_values_present = (
        negative_value_count > 0
    )

    zero_values_present = (
        zero_value_count > 0
    )


    balance_sheet_records = [
        record

        for record
        in extracted_records

        if (
            isinstance(
                record,
                dict
            )
            and record.get(
                "Category"
            )
            == "Balance sheet"
        )
    ]


    balance_sheet_period_valid = all(
        record.get(
            "Reporting Period"
        )
        == (
            "September 30, 2023 "
            "and June 30, 2023"
        )

        for record
        in balance_sheet_records
    )


    financial_reconciliation_records = [
        record

        for record
        in extracted_records

        if (
            isinstance(
                record,
                dict
            )
            and record.get(
                "Category"
            )
            == (
                "Financial performance "
                "reconciliation"
            )
        )
    ]


    diluted_eps_reconciliation_records = [
        record

        for record
        in financial_reconciliation_records

        if (
            isinstance(
                record.get("Metric"),
                str
            )
            and record.get(
                "Metric"
            ).strip().casefold()
            == "diluted earnings per share".casefold()
        )
    ]


    diluted_eps_impact_valid = (
        len(
            diluted_eps_reconciliation_records
        )
        == 1

        and (
            diluted_eps_reconciliation_records[
                0
            ].get(
                "Constant Currency Impact"
            )
            == 0.02
        )
    )


    devices_selected_records = [
        record

        for record
        in extracted_records

        if (
            isinstance(
                record,
                dict
            )

            and record.get(
                "Category"
            )
            == (
                "Selected product and service "
                "reconciliation"
            )

            and record.get(
                "Business Area"
            )
            == "Devices"
        )
    ]


    devices_negative_growth_valid = (
        len(
            devices_selected_records
        )
        == 1

        and (
            devices_selected_records[
                0
            ].get(
                "GAAP YoY Change"
            )
            == -22
        )

        and (
            devices_selected_records[
                0
            ].get(
                "Constant Currency YoY Change"
            )
            == -22
        )
    )


    CONTENT_DIAGNOSTICS = {
        "numeric_value_count":
            len(
                numeric_values
            ),

        "negative_value_count":
            negative_value_count,

        "zero_value_count":
            zero_value_count,

        "negative_values_present":
            negative_values_present,

        "zero_values_present":
            zero_values_present,

        "balance_sheet_record_count":
            len(
                balance_sheet_records
            ),

        "balance_sheet_period_valid":
            balance_sheet_period_valid,

        "diluted_eps_constant_currency_impact_valid":
            diluted_eps_impact_valid,

        "devices_negative_growth_valid":
            devices_negative_growth_valid
    }


else:

    CONTENT_DIAGNOSTICS = {
        "numeric_value_count":
            None,

        "negative_value_count":
            None,

        "zero_value_count":
            None,

        "negative_values_present":
            None,

        "zero_values_present":
            None,

        "balance_sheet_record_count":
            None,

        "balance_sheet_period_valid":
            None,

        "diluted_eps_constant_currency_impact_valid":
            None,

        "devices_negative_growth_valid":
            None
    }


print(
    json.dumps(
        CONTENT_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)

In [ ]:
# ============================================================
# 16. Structural evaluability and technical diagnostic summary
# ============================================================

part_structure_validity = {}


for part_number, result in (
    part_parsing_results.items()
):

    part_structure_validity[
        str(
            part_number
        )
    ] = all([
        result[
            "json_valid"
        ],
        result[
            "top_level_object_valid"
        ],
        result[
            "document_id_present"
        ],
        result[
            "document_id_correct"
        ],
        result[
            "branch_present"
        ],
        result[
            "branch_correct"
        ],
        result[
            "part_present"
        ],
        result[
            "part_correct"
        ],
        result[
            "records_present"
        ],
        result[
            "records_is_list"
        ]
    ])


top_level_structure_valid = all(
    part_structure_validity.values()
)


structurally_evaluable = all([
    top_level_structure_valid,
    record_schema_valid is True,
    field_types_valid is True
])


CONTENT_DIAGNOSTICS.update({
    "record_count_matches_reference":
        record_count_valid,

    "categories_valid":
        categories_valid,

    "category_counts_match_reference":
        category_counts_valid,

    "mandatory_fields_complete":
        mandatory_fields_complete,

    "missing_mandatory_value_count":
        missing_mandatory_value_count,

    "duplicate_record_key_count":
        duplicate_record_key_count,

    "duplicate_record_keys_absent":
        duplicate_record_keys_absent
})


TECHNICAL_DIAGNOSTICS = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_representation":
        INPUT_REPRESENTATION,

    "all_parts_json_valid":
        all_parts_json_valid,

    "all_parts_records_evaluable":
        all_parts_records_evaluable,

    "part_structure_validity":
        part_structure_validity,

    "part_parsing_results":
        PART_EXECUTION_SUMMARY,

    "top_level_structure_valid":
        top_level_structure_valid,

    "record_schema_valid":
        record_schema_valid,

    "records_with_structure_issues":
        records_with_structure_issues,

    "record_structure_issues":
        (
            record_structure_issues
            if all_parts_records_evaluable
            else None
        ),

    "field_types_valid":
        field_types_valid,

    "records_with_type_issues":
        records_with_type_issues,

    "field_type_issue_count":
        (
            len(
                field_type_issues
            )
            if all_parts_records_evaluable
            else None
        ),

    "field_type_issues":
        (
            field_type_issues
            if all_parts_records_evaluable
            else None
        ),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_match":
        category_counts_valid,

    "missing_mandatory_value_count":
        missing_mandatory_value_count,

    "missing_mandatory_values":
        (
            missing_mandatory_values
            if all_parts_records_evaluable
            else None
        ),

    "duplicate_record_key_count":
        duplicate_record_key_count,

    "duplicate_record_keys":
        duplicate_record_keys,

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "structurally_evaluable":
        bool(structurally_evaluable)
}


TECHNICAL_DIAGNOSTICS_PATH.write_text(
    json.dumps(
        TECHNICAL_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        TECHNICAL_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)

In [ ]:
# ============================================================
# 17. Experiment metadata
# ============================================================

PROMPT_FILE_METADATA = {
    str(
        part_number
    ): {
        "file":
            PART_PROMPT_PATHS[
                part_number
            ].name,

        "sha256":
            sha256_file(
                PART_PROMPT_PATHS[
                    part_number
                ]
            )
    }

    for part_number
    in sorted(
        PART_PROMPT_PATHS
    )
}


EXPERIMENT_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_format":
        SOURCE_FORMAT,

    "source_sha256":
        SOURCE_SHA256,

    "source_structure": {
        "expected_page_count":
            EXPECTED_PAGE_COUNT,

        "observed_page_count":
            PAGE_COUNT,

        "page_count_verified":
            PAGE_COUNT_VALID,

        "machine_readable_text_layer":
            TEXT_EXTRACTABLE,

        "expected_components_verified":
            DOCUMENT_GROUNDING_VALID
    },

    "input_representation":
        INPUT_REPRESENTATION,

    "direct_document_ingestion":
        DIRECT_DOCUMENT_INGESTION,

    "diagnostic_text_extraction_applied":
        True,

    "text_extraction_used_as_model_input":
        False,

    "pdf_to_text_conversion_applied":
        False,

    "ocr_applied":
        False,

    "page_cropping_applied":
        False,

    "page_extraction_applied":
        False,

    "layout_reconstruction_applied":
        False,

    "table_conversion_applied":
        False,

    "structural_conversion_applied":
        False,

    "normalisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_conversion_applied":
        False,

    "manual_correction_applied":
        False,

    "split_extraction_applied":
        True,

    "split_part_count":
        5,

    "split_strategy":
        "Deterministic source-section partition",

    "same_split_required_across_branches":
        True,

    "expected_extraction_scope": {
        "expected_record_count":
            EXPECTED_RECORD_COUNT,

        "expected_category_counts":
            EXPECTED_CATEGORY_COUNTS,

        "expected_part_counts":
            EXPECTED_PART_COUNTS,

        "expected_fields":
            EXPECTED_FIELDS
    },

    "reference_expectations_disclosed_to_model":
        False,

    "input_integrity_file":
        INPUT_INTEGRITY_PATH.name,

    "input_integrity_passed":
        INPUT_INTEGRITY_PASSED,

    "representation_file":
        REPRESENTATION_PATH.name,

    "prompt_files":
        PROMPT_FILE_METADATA,

    "raw_response_files":
        PART_RAW_RESPONSE_METADATA,

    "part_execution_summary_file":
        PART_EXECUTION_SUMMARY_PATH.name,

    "part_parsing_results":
        PART_EXECUTION_SUMMARY,

    "all_parts_json_valid":
        all_parts_json_valid,

    "all_parts_records_evaluable":
        all_parts_records_evaluable,

    "combined_parsed_extraction_file":
        (
            COMBINED_EXTRACTION_PATH.name
            if combined_extraction_created
            else None
        ),

    "combined_parsed_extraction_sha256":
        combined_extraction_sha256,

    "expected_output_format":
        (
            "Five JSON objects with document_id, "
            "branch, part and records"
        ),

    "execution_environment":
        "Five independent ChatGPT conversations",

    "observed_record_count":
        observed_record_count,

    "observed_category_counts":
        observed_category_counts,

    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,

    "structurally_evaluable":
        bool(structurally_evaluable),

    "notes": (
        "Branch A supplies the complete original D6 PDF "
        "directly to the model in five predefined source-section "
        "extraction runs. The partition is defined before model "
        "execution and must be held constant across Branches A, "
        "B and C. Stage 1 reference counts are retained only for "
        "post-extraction diagnostics and are not disclosed to the "
        "model. Raw responses are preserved before parsing. "
        "Accuracy is evaluated separately in Validation A — D6."
    )
}


EXPERIMENT_METADATA_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        EXPERIMENT_METADATA,
        ensure_ascii=False,
        indent=2
    )
)

In [ ]:
# ============================================================
# 18. Experiment summary
# ============================================================

EXPERIMENT_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        INPUT_INTEGRITY_PASSED,

    "input_representation":
        INPUT_REPRESENTATION,

    "direct_document_ingestion":
        DIRECT_DOCUMENT_INGESTION,

    "split_extraction_applied":
        True,

    "split_part_count":
        5,

    "all_parts_json_valid":
        all_parts_json_valid,

    "all_parts_records_evaluable":
        all_parts_records_evaluable,

    "structurally_evaluable":
        bool(
            structurally_evaluable
        ),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_match":
        category_counts_valid,

    "record_schema_valid":
        record_schema_valid,

    "records_with_structure_issues":
        records_with_structure_issues,

    "field_types_valid":
        field_types_valid,

    "records_with_type_issues":
        records_with_type_issues,

    "missing_mandatory_value_count":
        missing_mandatory_value_count,

    "duplicate_record_key_count":
        duplicate_record_key_count,

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "raw_responses_preserved":
        all(
            path.exists()

            for path
            in PART_RAW_RESPONSE_PATHS.values()
        ),

    "combined_parsed_extraction_created":
        combined_extraction_created,

    "content_validation_performed":
        False,

    "notes": (
        "This notebook performs source verification, "
        "D6 Branch A direct-PDF extraction preservation "
        "and technical/schema checks. Agreement with the "
        "fixed Stage 1 reference is evaluated separately "
        "in Validation A — D6."
    )
}


EXPERIMENT_SUMMARY_PATH.write_text(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    )
)

In [ ]:
# ============================================================
# 19. Final experiment summary
# ============================================================

print(
    "=" * 60
)

print(
    "D6 Branch A experiment completed"
)

print(
    "=" * 60
)


print(
    "Input integrity passed       :",
    INPUT_INTEGRITY_PASSED
)

print(
    "All five responses preserved:",
    all(
        path.exists()

        for path
        in PART_RAW_RESPONSE_PATHS.values()
    )
)

print(
    "All parts valid JSON         :",
    all_parts_json_valid
)

print(
    "All parts records evaluable  :",
    all_parts_records_evaluable
)

print(
    "Expected records             :",
    EXPECTED_RECORD_COUNT
)

print(
    "Observed records             :",
    (
        observed_record_count

        if all_parts_records_evaluable

        else "Not fully evaluable"
    )
)

print(
    "Record count matches         :",
    record_count_valid
)

print(
    "Category counts match        :",
    category_counts_valid
)

print(
    "Record schema valid          :",
    record_schema_valid
)

print(
    "Field types valid            :",
    field_types_valid
)

print(
    "Structurally evaluable       :",
    structurally_evaluable
)

print(
    "Content validation performed : False"
)

print(
    "Next step: Validation A — D6"
)

In [ ]:
# ============================================================
# 20. Final artefact inventory
# ============================================================

GENERATED_OUTPUTS = [
    INPUT_INTEGRITY_PATH,
    REPRESENTATION_PATH,
    PART_EXECUTION_SUMMARY_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    EXPERIMENT_METADATA_PATH,
    EXPERIMENT_SUMMARY_PATH
]


GENERATED_OUTPUTS.extend(
    PART_PROMPT_PATHS[
        part_number
    ]

    for part_number
    in sorted(
        PART_PROMPT_PATHS
    )
)


GENERATED_OUTPUTS.extend(
    PART_RAW_RESPONSE_PATHS[
        part_number
    ]

    for part_number
    in sorted(
        PART_RAW_RESPONSE_PATHS
    )
)


if (
    combined_extraction_created
    and COMBINED_EXTRACTION_PATH.exists()
):

    GENERATED_OUTPUTS.append(
        COMBINED_EXTRACTION_PATH
    )


print(
    "Generated D6 Branch A files:\n"
)


for output_path in GENERATED_OUTPUTS:

    print(
        "-",
        output_path.name,
        "| exists:",
        output_path.exists()
    )

In [ ]:
# ============================================================
# 21. Download experiment artefacts
# ============================================================

for output_path in GENERATED_OUTPUTS:

    if output_path.exists():

        files.download(
            str(
                output_path
            )
        )